# Canonical Model 02 · Visual Diagnostics

The canonical forcing is deliberately **visible**: a regional down-valley
gradient, a high-K paleochannel, a cross-valley low-K constriction, gaining and
losing stream reaches, a perched lake, an **infiltration** mound, and pumping
cones. Each plot below is rendered in **both** backends — static `matplotlib`
and interactive `plotly` — so the same diagnostic works in a report and on screen.

> **Visual standard:** a plot should reveal a decision-relevant feature, not
> merely prove that plotting code runs.

In [ ]:
import sys
from pathlib import Path

# Make the in-repo `src/` importable when myflopy is not pip-installed.
src = Path.cwd().parents[2] / 'src'
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))
import myflopy as mf
from canonical_notebook_style import notebook_header

notebook_header('02', 'Visual Diagnostics', 'See gradients, the paleochannel, gaining/losing streams, the perched lake, and pumping.')

root = Path('../artifacts/canonical_visuals')
config = mf.CanonicalModelConfig.validation()
model = mf.build_canonical_model(root / 'gwf', config=config)
assert model.run_simulation()[0]

import numpy as np
import matplotlib.pyplot as plt
from myflopy.modflow.mf6.grid.plotting import build_choropleth

# Two built-ins carry every map below -- no notebook-local helpers:
#   model.hds.array(layer=, per=)          -> one head value per Voronoi cell
#   build_choropleth(model.vor, custom_zs=).plot_mpl()  -> static matplotlib
#   build_choropleth(model.vor, custom_zs=).plot()       -> interactive plotly
# Pass model= to outline named regions (streams, lake) on the matplotlib map.


## 1 · Water table and regional gradient

**What to look for:** a strong head decline from the up-valley head toward the
valley mouth (left to right), bent by the high-K paleochannel along the axis and
tightened at the mid-valley constriction. The stream and lake outlines mark where
surface water interacts with this water table.

In [ ]:
wt = model.hds.array(layer=0)   # final-period head, one value per cell
wt_map = build_choropleth(model.vor, model=model, custom_zs=list(wt), layer=0)
wt_map.plot_mpl(title='Upper unconfined water table (ft) - final period',
                cmap='viridis', outline_regions=('all_streams', 'all_lakes'))
wt_map.plot()   # interactive plotly version of the same map

## 2 · Vertical structure (four-layer head mosaic)

**What to look for:** the two unconfined layers track each other; the head field
changes character across the **aquitard** (layer 3) into the **confined** aquifer
(layer 4), where the deep pumping cone is most distinct. This is the visual proof
the low-K aquitard damps vertical communication.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
roles = ['L1 upper unconfined', 'L2 lower unconfined', 'L3 aquitard', 'L4 confined']
for layer, (ax, role) in enumerate(zip(axes.ravel(), roles)):
    build_choropleth(model.vor, custom_zs=list(model.hds.array(layer=layer)), layer=layer).plot_mpl(ax=ax, title=role)
fig.suptitle('Head by layer (final period)', fontsize=13)
fig.tight_layout()

## 3 · Gaining vs. losing streams

The SFR network is a resolved, continuously wet stream, not a decorative line.
**What to look for:** stage held just above the streambed along the whole channel,
and the stream→aquifer exchange flipping sign — **negative (gaining)** where the
stream sits below the water table in the headwaters, turning **positive (losing)**
downgradient toward the perched lake.

In [ ]:
per = config.nper - 1
profile = model.packages.sfr.results.profile(per=per)

# The figure colors each reach by sign -- blue gains, red loses. Count them
# too, so the diagnostic is a number and not only a picture.
q = profile.get()['q'].to_numpy(float)
print(f'{(q < 0).sum()} gaining / {(q > 0).sum()} losing reaches')

profile.plot(plot_fig=True)

## 4 · The perched lake and its budget

The terminal lake receives the main stem's routed flow (via MVR) and sits above
the downgradient water table. **What to look for:** the lake-aquifer exchange is
**negative** — the perched lake **loses** water downward to the aquifer through
its bed (both horizontal and vertical connections), rather than the aquifer
feeding the lake.

In [ ]:
display(model.packages.lak.results.q.budget_summary(per=per))
# Signed exchange by connection type for this period (matplotlib):
model.packages.lak.results.q.plot_budget(per=per)

## 5 · Hydraulic conductivity — the paleochannel

**What to look for:** a high-K ribbon (the buried **paleochannel**) running down
the valley axis that channels flow toward the streams and lake, pinched by the
low-K cross-valley **constriction** near mid-valley. This is the heterogeneity the
PEST notebooks (04–06) try to recover from heads alone.

In [ ]:
k0 = np.asarray(model.gwf.npf.k.get_data(), float)[0]
k_map = build_choropleth(model.vor, custom_zs=list(k0), layer=0)
k_map.plot_mpl(cmap='cividis', title='Layer 1 hydraulic conductivity (ft/day)')
k_map.plot()

## 6 · Pumping drawdown

**What to look for:** a localized cone of depression in the lower unconfined
aquifer (layer 2) around the shallow well, where head has dropped from the start
of the simulation to the end. The confined well (layer 4) produces a separate,
broader cone beneath the aquitard.

In [ ]:
times = model.hds.kstpkper
drawdown = model.hds.array(layer=1, kstpkper=times[0]) - model.hds.array(layer=1, kstpkper=times[-1])
build_choropleth(model.vor, model=model, custom_zs=list(drawdown), layer=1).plot_mpl(
    cmap='magma_r', title='Layer 2 drawdown (ft): start - final',
    outline_regions=('all_streams', 'all_lakes'))

## Interpretation checklist

- Contours bend along the paleochannel and tighten at the constriction.
- The stream is gaining in the headwaters and losing toward the lake.
- The perched lake leaks downward (negative exchange).
- The aquitard damps vertical propagation between the unconfined and confined layers.
- Each pumping well shows a localized drawdown cone.

Continue to **03 · PRT and Parallel** to test movement and computational scaling.